# Extended Thermosurvey Data Preparation

This notebook prepares a new dataset for the following ShadeAdapt experiments:

- **Experiment 2:** temperature + relative humidity + personal outdoor heat threshold (`e2_2`)
- **Experiment 5:** temperature + relative humidity + historical wind + solar radiation
- **Experiment 6:** Experiment 5 + personal outdoor heat threshold
- **Experiment 7:** use all valid indoor/semi-outdoor/outdoor observations, with location as a modelling factor, while evaluating the final use case on outdoor observations

## Important design decisions

1. This notebook **does not fetch historical wind or solar radiation**. The source CSV does not contain those variables.
2. It keeps `city`, `interview_date`, `thermal_measurement_time`, and `timezone` so a separate historical-weather enrichment step can fetch wind/solar later.
3. It keeps **all valid location types** (`e7 = 1, 2, 3`) instead of filtering to outdoor rows, because Experiment 7 needs indoor and outdoor data.
4. It keeps `e2_2` as `personal_heat_threshold_c`. Rows with missing `e2_2` are retained because Experiments 5 and 7 do not require it.
5. `e3` remains the thermal-sensation target:
   - 1–5 → Not warm / OK
   - 6–7 → Warm
   - 8–9 → Too hot
   - 98/99 → removed


In [1]:
from pathlib import Path
import pandas as pd

print("Pandas version:", pd.__version__)
print("Current working directory:", Path.cwd())

# Locate the raw dataset whether the notebook is run from:
# - ai/thermal_model/preprocessing/
# - ai/thermal_model/
# - repository root
cwd = Path.cwd()

possible_paths = [
    cwd / "data" / "raw" / "Data.csv",
    cwd.parent / "data" / "raw" / "Data.csv",
    cwd / "ai" / "thermal_model" / "data" / "raw" / "Data.csv",
]

DATA_PATH = None
for path in possible_paths:
    path = path.resolve()
    if path.exists():
        DATA_PATH = path
        break

print("Dataset path:", DATA_PATH)
print("Dataset found:", DATA_PATH is not None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find Data.csv. Expected it under "
        "ai/thermal_model/data/raw/Data.csv"
    )


Pandas version: 2.3.3
Current working directory: ai/thermal_model\preprocessing
Dataset path: ai/thermal_model\data\raw\Data.csv
Dataset found: True


## Load the raw dataset

The source file contains Polish/European text formatting, so `cp1250` is used. `low_memory=False` avoids mixed-type warnings while loading the 266-column file.


In [2]:
df = pd.read_csv(
    DATA_PATH,
    encoding="cp1250",
    low_memory=False
)

print("Raw shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

required_columns = [
    "nr",
    "lokalizacja",
    "data",
    "godzina",
    "e10",
    "e7",
    "e2_2",
    "e3",
    "e9t",
    "e9w",
]

missing_required = [c for c in required_columns if c not in df.columns]

print("\nRequired columns present:", len(missing_required) == 0)
print("Missing required columns:", missing_required)

if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")


Raw shape: (2111, 266)
Rows: 2111
Columns: 266

Required columns present: True
Missing required columns: []


## Convert the environmental and personal-threshold variables

The source data use commas as decimal separators in several numeric columns.


In [3]:
def european_numeric(series):
    return pd.to_numeric(
        series.astype("string").str.replace(",", ".", regex=False),
        errors="coerce"
    )

temperature_c = european_numeric(df["e9t"])
relative_humidity = european_numeric(df["e9w"])
personal_heat_threshold_c = european_numeric(df["e2_2"])

print("Temperature conversion failures:",
      (df["e9t"].notna() & temperature_c.isna()).sum())

print("Humidity conversion failures:",
      (df["e9w"].notna() & relative_humidity.isna()).sum())

print("Personal threshold conversion failures:",
      (df["e2_2"].notna() & personal_heat_threshold_c.isna()).sum())

print("\nPersonal outdoor heat threshold availability:")
print("Available:", personal_heat_threshold_c.notna().sum())
print("Missing:", personal_heat_threshold_c.isna().sum())

print("\nPersonal threshold range:")
print(personal_heat_threshold_c.min(), "to", personal_heat_threshold_c.max())


Temperature conversion failures: 0
Humidity conversion failures: 0
Personal threshold conversion failures: 0

Personal outdoor heat threshold availability:
Available: 1840
Missing: 271

Personal threshold range:
20.0 to 57.2


## Keep valid thermal-sensation observations

Unlike the original preprocessing notebook, this notebook **does not filter by `e7`**.

This is intentional:
- Experiments 2, 5 and 6 can filter to `e7 in {2, 3}` later.
- Experiment 7 requires the indoor observations as additional real training data.


In [4]:
valid_mask = (
    df["e3"].between(1, 9)
    & temperature_c.notna()
    & relative_humidity.notna()
)

prepared_df = df.loc[
    valid_mask,
    [
        "nr",
        "lokalizacja",
        "data",
        "godzina",
        "e10",
        "e7",
        "e2_2",
        "e3",
    ]
].copy()

prepared_df["temperature_c"] = temperature_c.loc[valid_mask].astype(float)
prepared_df["relative_humidity"] = relative_humidity.loc[valid_mask].astype(float)
prepared_df["personal_heat_threshold_c"] = (
    personal_heat_threshold_c.loc[valid_mask]
)

print("Valid rows retained:", len(prepared_df))
print("\nLocation distribution:")
print(prepared_df["e7"].value_counts().sort_index())


Valid rows retained: 2099

Location distribution:
e7
1    1410
2      62
3     627
Name: count, dtype: int64


## Create city and historical-weather lookup fields

`e10` is retained as the thermal-section measurement-time field and is used to build `thermal_datetime_local`. `godzina` is retained separately as the interview start time for traceability.

The final CSV includes the city timezone so a later weather-enrichment notebook can request historical data using the correct local time.


In [5]:
prepared_df["city_code"] = prepared_df["lokalizacja"].astype(int)

prepared_df["city"] = prepared_df["city_code"].map({
    1: "Warsaw",
    2: "Madrid",
})

prepared_df["timezone"] = prepared_df["city_code"].map({
    1: "Europe/Warsaw",
    2: "Europe/Madrid",
})

prepared_df["interview_date"] = pd.to_datetime(
    prepared_df["data"],
    format="%m/%d/%Y",
    errors="coerce"
).dt.strftime("%Y-%m-%d")

prepared_df["interview_start_time"] = pd.to_datetime(
    prepared_df["godzina"],
    format="%H:%M:%S",
    errors="coerce"
).dt.strftime("%H:%M:%S")

prepared_df["thermal_measurement_time"] = pd.to_datetime(
    prepared_df["e10"],
    format="%H:%M:%S",
    errors="coerce"
).dt.strftime("%H:%M:%S")

prepared_df["thermal_datetime_local"] = pd.to_datetime(
    prepared_df["interview_date"]
    + " "
    + prepared_df["thermal_measurement_time"],
    errors="coerce"
).dt.strftime("%Y-%m-%d %H:%M:%S")

print("Date parse failures:",
      prepared_df["interview_date"].isna().sum())

print("Thermal measurement time parse failures:",
      prepared_df["thermal_measurement_time"].isna().sum())

print("Combined datetime failures:",
      prepared_df["thermal_datetime_local"].isna().sum())

display(
    prepared_df[
        [
            "nr",
            "city",
            "interview_date",
            "interview_start_time",
            "thermal_measurement_time",
            "thermal_datetime_local",
        ]
    ].head(10)
)


Date parse failures: 0
Thermal measurement time parse failures: 0
Combined datetime failures: 0


,nr,city,interview_date,interview_start_time,thermal_measurement_time,thermal_datetime_local
0,4,Warsaw,2022-07-07,13:21:46,13:27:22,2022-07-07 13:27:22
1,5,Warsaw,2022-07-07,14:22:44,14:27:46,2022-07-07 14:27:46
2,8,Warsaw,2022-07-07,16:12:34,16:21:30,2022-07-07 16:21:30
3,9,Warsaw,2022-07-07,16:15:17,16:28:03,2022-07-07 16:28:03
4,10,Warsaw,2022-07-07,17:01:42,17:10:15,2022-07-07 17:10:15
5,24,Warsaw,2022-07-08,13:17:15,13:25:33,2022-07-08 13:25:33
6,25,Warsaw,2022-07-08,14:01:30,14:20:11,2022-07-08 14:20:11
7,27,Warsaw,2022-07-08,15:30:38,15:36:52,2022-07-08 15:36:52
8,47,Warsaw,2022-07-09,13:51:58,14:14:43,2022-07-09 14:14:43
9,53,Warsaw,2022-07-09,15:13:12,15:23:38,2022-07-09 15:23:38


## Create location indicators

For Experiment 7, the model can use location/domain information during training.

- `is_strict_outdoor = 1` only for `e7 = 3`
- `is_outdoor_or_semi = 1` for `e7 = 2 or 3`

The final evaluation for ShadeAdapt should still focus on the outdoor/semi-outdoor deployment subset.


In [6]:
prepared_df["location_type"] = prepared_df["e7"].map({
    1: "Indoor",
    2: "Balcony/Terrace",
    3: "Outdoor",
})

prepared_df["is_strict_outdoor"] = (
    prepared_df["e7"] == 3
).astype(int)

prepared_df["is_outdoor_or_semi"] = (
    prepared_df["e7"].isin([2, 3])
).astype(int)

prepared_df["has_personal_heat_threshold"] = (
    prepared_df["personal_heat_threshold_c"].notna()
).astype(int)

print(
    prepared_df[
        [
            "e7",
            "location_type",
            "is_strict_outdoor",
            "is_outdoor_or_semi",
        ]
    ]
    .drop_duplicates()
    .sort_values("e7")
)


     e7    location_type  is_strict_outdoor  is_outdoor_or_semi
694   1           Indoor                  0                   0
632   2  Balcony/Terrace                  0                   1
0     3          Outdoor                  1                   1


## Create the three-class ordinal target


In [7]:
def map_thermal_class(e3_value):
    if 1 <= e3_value <= 5:
        return "Not warm / OK"
    elif 6 <= e3_value <= 7:
        return "Warm"
    elif 8 <= e3_value <= 9:
        return "Too hot"
    return None

prepared_df["thermal_class"] = (
    prepared_df["e3"].apply(map_thermal_class)
)

prepared_df["thermal_class_code"] = (
    prepared_df["thermal_class"]
    .map({
        "Not warm / OK": 0,
        "Warm": 1,
        "Too hot": 2,
    })
    .astype(int)
)

print("Target distribution:")
print(
    prepared_df["thermal_class"]
    .value_counts()
)

print("\nTarget code distribution:")
print(
    prepared_df["thermal_class_code"]
    .value_counts()
    .sort_index()
)


Target distribution:
thermal_class
Warm             897
Not warm / OK    784
Too hot          418
Name: count, dtype: int64

Target code distribution:
thermal_class_code
0    784
1    897
2    418
Name: count, dtype: int64


## Create the final extended modelling dataset

This CSV deliberately keeps rows with missing `personal_heat_threshold_c`.

Use complete cases for Experiments 2 and 6, but do not throw those rows away globally because Experiments 5 and 7 can still use them.


In [8]:
final_columns = [
    "nr",
    "city_code",
    "city",
    "timezone",
    "interview_date",
    "interview_start_time",
    "thermal_measurement_time",
    "thermal_datetime_local",
    "e7",
    "location_type",
    "is_strict_outdoor",
    "is_outdoor_or_semi",
    "temperature_c",
    "relative_humidity",
    "personal_heat_threshold_c",
    "has_personal_heat_threshold",
    "e3",
    "thermal_class",
    "thermal_class_code",
]

prepared_df = (
    prepared_df[final_columns]
    .reset_index(drop=True)
)

print("Final shape:", prepared_df.shape)
print("\nMissing values:")
print(prepared_df.isna().sum())

display(prepared_df.head(10))


Final shape: (2099, 19)

Missing values:
nr                               0
city_code                        0
city                             0
timezone                         0
interview_date                   0
interview_start_time             0
thermal_measurement_time         0
thermal_datetime_local           0
e7                               0
location_type                    0
is_strict_outdoor                0
is_outdoor_or_semi               0
temperature_c                    0
relative_humidity                0
personal_heat_threshold_c      268
has_personal_heat_threshold      0
e3                               0
thermal_class                    0
thermal_class_code               0
dtype: int64


,nr,city_code,city,timezone,interview_date,interview_start_time,thermal_measurement_time,thermal_datetime_local,e7,location_type,is_strict_outdoor,is_outdoor_or_semi,temperature_c,relative_humidity,personal_heat_threshold_c,has_personal_heat_threshold,e3,thermal_class,thermal_class_code
0,4,1,Warsaw,Europe/Warsaw,2022-07-07,13:21:46,13:27:22,2022-07-07 13:27:22,3,Outdoor,1,1,27.5,29.6,25.0,1,7,Warm,1
1,5,1,Warsaw,Europe/Warsaw,2022-07-07,14:22:44,14:27:46,2022-07-07 14:27:46,3,Outdoor,1,1,23.5,37.7,25.0,1,6,Warm,1
2,8,1,Warsaw,Europe/Warsaw,2022-07-07,16:12:34,16:21:30,2022-07-07 16:21:30,3,Outdoor,1,1,23.9,53.5,27.0,1,5,Not warm / OK,0
3,9,1,Warsaw,Europe/Warsaw,2022-07-07,16:15:17,16:28:03,2022-07-07 16:28:03,3,Outdoor,1,1,27.8,36.6,29.0,1,5,Not warm / OK,0
4,10,1,Warsaw,Europe/Warsaw,2022-07-07,17:01:42,17:10:15,2022-07-07 17:10:15,3,Outdoor,1,1,27.8,36.6,29.0,1,5,Not warm / OK,0
5,24,1,Warsaw,Europe/Warsaw,2022-07-08,13:17:15,13:25:33,2022-07-08 13:25:33,3,Outdoor,1,1,27.2,57.6,28.0,1,7,Warm,1
6,25,1,Warsaw,Europe/Warsaw,2022-07-08,14:01:30,14:20:11,2022-07-08 14:20:11,3,Outdoor,1,1,27.1,51.7,<NA>,0,5,Not warm / OK,0
7,27,1,Warsaw,Europe/Warsaw,2022-07-08,15:30:38,15:36:52,2022-07-08 15:36:52,3,Outdoor,1,1,25.0,70.8,<NA>,0,5,Not warm / OK,0
8,47,1,Warsaw,Europe/Warsaw,2022-07-09,13:51:58,14:14:43,2022-07-09 14:14:43,3,Outdoor,1,1,20.9,51.8,<NA>,0,5,Not warm / OK,0
9,53,1,Warsaw,Europe/Warsaw,2022-07-09,15:13:12,15:23:38,2022-07-09 15:23:38,3,Outdoor,1,1,22.1,50.2,<NA>,0,5,Not warm / OK,0


## Check the four planned experiments

This cell does not train models. It only checks how many real observations are available for each experiment.

Experiments 5 and 6 are marked as **pre-enrichment** because historical wind and solar radiation still need to be added in a separate step.


In [9]:
# Experiment 2:
# outdoor/semi-outdoor + temperature + RH + personal threshold
exp2 = prepared_df[
    (prepared_df["is_outdoor_or_semi"] == 1)
    & prepared_df["personal_heat_threshold_c"].notna()
].copy()

# Experiment 5:
# outdoor/semi-outdoor + temp/RH; wind/solar will be joined later
exp5_pre_enrichment = prepared_df[
    prepared_df["is_outdoor_or_semi"] == 1
].copy()

# Experiment 6:
# Experiment 5 + personal threshold
exp6_pre_enrichment = prepared_df[
    (prepared_df["is_outdoor_or_semi"] == 1)
    & prepared_df["personal_heat_threshold_c"].notna()
].copy()

# Experiment 7:
# all valid indoor/semi/outdoor real observations
exp7 = prepared_df.copy()

summary = pd.DataFrame({
    "experiment": [
        "Experiment 2",
        "Experiment 5 (pre-enrichment)",
        "Experiment 6 (pre-enrichment)",
        "Experiment 7",
    ],
    "rows": [
        len(exp2),
        len(exp5_pre_enrichment),
        len(exp6_pre_enrichment),
        len(exp7),
    ],
    "ready_to_train_now": [
        True,
        False,
        False,
        True,
    ],
    "reason_if_not_ready": [
        "",
        "Historical wind and solar radiation still need to be added.",
        "Historical wind and solar radiation still need to be added.",
        "",
    ],
})

display(summary)

print("\nExperiment 7 location distribution:")
print(exp7["location_type"].value_counts())

print("\nOutdoor/semi-outdoor target distribution:")
print(
    exp5_pre_enrichment["thermal_class"]
    .value_counts()
)

print("\nExperiment 2 target distribution:")
print(
    exp2["thermal_class"]
    .value_counts()
)


,experiment,rows,ready_to_train_now,reason_if_not_ready
0,Experiment 2,596,True,
1,Experiment 5 (pre-enrichment),689,False,Historical wind and solar radiation still need...
2,Experiment 6 (pre-enrichment),596,False,Historical wind and solar radiation still need...
3,Experiment 7,2099,True,



Experiment 7 location distribution:
location_type
Indoor             1410
Outdoor             627
Balcony/Terrace      62
Name: count, dtype: int64

Outdoor/semi-outdoor target distribution:
thermal_class
Not warm / OK    246
Warm             230
Too hot          213
Name: count, dtype: int64

Experiment 2 target distribution:
thermal_class
Not warm / OK    205
Warm             196
Too hot          195
Name: count, dtype: int64


## Save the new prepared CSV


In [10]:
PROCESSED_DIR = DATA_PATH.parent.parent / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    PROCESSED_DIR
    / "thermosurvey_extended_prepared.csv"
)

prepared_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved to:")
print(OUTPUT_PATH)
print("\nFile exists:", OUTPUT_PATH.exists())

# Verify the saved file.
check_df = pd.read_csv(OUTPUT_PATH)

print("\nReloaded shape:", check_df.shape)
print("Reloaded columns:", check_df.columns.tolist())


Saved to:
ai/thermal_model\data\processed\thermosurvey_extended_prepared.csv

File exists: True

Reloaded shape: (2099, 19)
Reloaded columns: ['nr', 'city_code', 'city', 'timezone', 'interview_date', 'interview_start_time', 'thermal_measurement_time', 'thermal_datetime_local', 'e7', 'location_type', 'is_strict_outdoor', 'is_outdoor_or_semi', 'temperature_c', 'relative_humidity', 'personal_heat_threshold_c', 'has_personal_heat_threshold', 'e3', 'thermal_class', 'thermal_class_code']


## Next step: historical weather enrichment

For Experiments 5 and 6, use the following fields from this prepared CSV:

- `city`
- `timezone`
- `thermal_datetime_local`

and retrieve historical weather close to that timestamp, including at minimum:

- wind speed
- solar radiation

Do not generate synthetic wind/solar values. They should come from a historical weather/reanalysis source.

After enrichment, merge the weather values back by respondent/timestamp and then run Experiments 5 and 6.
